In [46]:
# Importing the python libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV  

In [47]:
import pandas as pd  

cleaned_customer_transaction_data = pd.read_csv(r"C:/U sers/USER/Desktop/AMDARI/Fraudulent_Transaction_Detection_For_Finlora_Company/Fraudulent_Transaction_Detection_For_Finlora_Company/Finlora Dataset/artifacts/EDA_Data.csv")  

FileNotFoundError: [Errno 2] No such file or directory: 'C:/U sers/USER/Desktop/AMDARI/Fraudulent_Transaction_Detection_For_Finlora_Company/Fraudulent_Transaction_Detection_For_Finlora_Company/Finlora Dataset/artifacts/EDA_Data.csv'

## Threshold-based Risk Flags
To help our fraud detection model isolate suspicious behaviour, continius numerical columns were converted into binary risk flags ($1$ = High Risk, $0$ = Normal),

### Risk Indicator Breakdown
* 'late-night_hours: Flags off-peak transactions (00:00-05:00) when account holders are typically inactive.
* 'amount_high: Flags high-value transaction (> $1,000 USD) carrying greater finacial risk.
* 'high_ip_risk: Flags connections from suspicious IP range (score > 0.8).
* 'low_device_trust: Flags unverified or unrecognize hardware (score < 0.3).
* 'new_account & very_new_account: Flag accounts under 30 days and 7 days old to capture early-stage synthetic fraud.
* 'velocity_spike: Flags rapid-fir activity (> 3 transaction in 1 hour) indciating potential bot attacks.

### Output Interpretation
Because legitimate transaction drastically outnumber fraudulent ones, the majorityof output rows display '0's across most features. Isolated '1's (such as line 1 showing 'low_device_trust = 1') represent specific risk triggers for analysis. 

In [48]:
# Convert timestamp to datetime and extract the hour, day_of_week, and is_weekend 
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp'])

# Extract the datetime features safely
cleaned_customer_transaction_data['hour'] = cleaned_customer_transaction_data['timestamp'].dt.hour
cleaned_customer_transaction_data['is_weekend'] = cleaned_customer_transaction_data['day_of_week'].isin([5, 6]).astype(int)

 
# Creating a threshold based features from the following risk_signsl (Build feature logic)
cleaned_customer_transaction_data['late_night_hours'] = ((cleaned_customer_transaction_data['hour'] >=3) & (cleaned_customer_transaction_data['hour'] <=7)).astype(int)
cleaned_customer_transaction_data['amount_high'] = (cleaned_customer_transaction_data['amount_usd'] >=1000).astype(int)
cleaned_customer_transaction_data['high_ip_risk'] = (cleaned_customer_transaction_data['ip_risk_score'] >=0.8).astype(int)
cleaned_customer_transaction_data['low_device_trust'] = (cleaned_customer_transaction_data['device_trust_score'] >=0.5).astype(int)
cleaned_customer_transaction_data['new_account'] = ((cleaned_customer_transaction_data['account_age_days'] >=30) & (cleaned_customer_transaction_data['account_age_days'] <=90)).astype(int)
cleaned_customer_transaction_data['very_new_account'] = (cleaned_customer_transaction_data['account_age_days'] >=30).astype(int)
cleaned_customer_transaction_data['velocity_spike'] = (cleaned_customer_transaction_data['txn_velocity_1h'] >=3).astype(int)

# Select the features subset
high_risk_signal_features = cleaned_customer_transaction_data[
    ['late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account', 'velocity_spike']
]

# Dispaly output
high_risk_signal_features.head()               

,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,0,0,0,1,0,1,0
1,0,0,0,0,0,1,0
2,0,0,0,1,0,1,0
3,0,0,0,1,0,1,0
4,0,0,0,1,0,1,0


In [49]:
# Creating the features in our datasets
list(cleaned_customer_transaction_data.columns)    

['timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'new_device',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike',
 'days_of_week',
 'day_of_week',
 'is_weekend']

Feature Selection 

In [66]:
# Dropping all temporary buckets, identifiers column (lds), and some irrelvant variables columns

to_drop = ['account_age_bucket', 'device_trust_score_bucket', 'ip_risk_score_bucket', 'amount_usd_bucket',
            'transaction_id', 'customer_id', 'device_id', 'ip_addres s', 'chargeback_history_count', 'exchange_rate_src_to_dest'
]   
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(columns=to_drop, errors='ignore')  

In [67]:
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop( columns=['days_of_week'], errors ='ignore') 

In [ ]:
# cleaned_customer_transaction_data = cleaned_customer_transaction_data.dropna()  

  Defining categorical features

In [68]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object', 'bool']).columns
categorical_features       

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

 Defining numerical features

In [69]:
numerical_features = cleaned_customer_transaction_data.select_dtypes(include=['int', 'float']).columns.drop('is_fraud')
numerical_features        

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'late_night_hours',
       'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account',
       'very_new_account', 'velocity_spike', 'day_of_week', 'is_weekend'],
      dtype='object')

In [ ]:
print(f"categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")
print(f"Dataset: {cleaned_customer_transaction_data.shape}")           

categorical: 8
Numerical: 20
Dataset: (10731, 30)


In [62]:
cleaned_customer_transaction_data.columns 

Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account', 'velocity_spike',
       'day_of_week', 'is_weekend'],
      dtype='object')

In [63]:
cleaned_customer_transaction_data.info() 

<class 'pandas.core.frame.DataFrame'>
Index: 10731 entries, 0 to 10839
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10731 non-null  datetime64[ns, UTC]
 1   home_country         10731 non-null  object             
 2   source_currency      10731 non-null  object             
 3   dest_currency        10731 non-null  object             
 4   channel              10731 non-null  object             
 5   amount_src           10731 non-null  float64            
 6   amount_usd           10731 non-null  float64            
 7   fee                  10731 non-null  float64            
 8   new_device           10731 non-null  bool               
 9   ip_country           10731 non-null  object             
 10  location_mismatch    10731 non-null  bool               
 11  ip_risk_score        10731 non-null  float64            
 12  kyc_tier             10

In [72]:
cleaned_customer_transaction_data.to_csv(r"../Finlora Dataset/artifacts/Engineering_Data.csv", index=False)  